# module-composition — worked example 2: Same MLP via nn.Sequential

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-composition`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`nn.Sequential(l1, relu, l2)` is itself a Module that chains its children in order. Holding the whole pipeline behind one attribute `self.net` changes the parameter names (`net.0.weight` instead of `fc1.weight`) but preserves the parameter count and forward output.

## Worked solution

We rebuild the two-layer MLP as a single Sequential.

1. **Wrap the pipeline.** `self.net = nn.Sequential(Linear, ReLU, Linear)` registers as one child named `net`; its sub-children are indexed `0`, `1`, `2`.
2. **forward.** Just `self.net(x)` — Sequential runs each stage in order, so the explicit ReLU call disappears from forward.
3. **Naming.** Parameters are now `net.0.weight`, `net.0.bias`, `net.2.weight`, `net.2.bias` (index 1 is the parameter-free ReLU).
4. **Parity.** Same 4 parameters, same output shape as the named-attribute version.

The demo prints the Sequential length, the parameter names, and the output shape.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(1)

class SequentialMLP(nn.Module):
    def __init__(self, in_f, hid, out_f):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_f, hid),
            nn.ReLU(),
            nn.Linear(hid, out_f),
        )
    def forward(self, x):
        return self.net(x)

model = SequentialMLP(4, 8, 3)
print('sequential length:', len(model.net))
print('param names:', [n for n, _ in model.named_parameters()])
print('output shape:', tuple(model(t.randn(5, 4)).shape))